In [1]:
!pip install -q torch torchvision scikit-learn matplotlib pandas tqdm

In [2]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [3]:
import os
import zipfile

dataset_dir = "/content/datasets"

for name in ["train", "test"]:
    zip_path = os.path.join(dataset_dir, f"{name}.zip")
    extract_path = os.path.join(dataset_dir, name)

    os.makedirs(extract_path, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_path)

    print(f"\n{name.upper()} extracted successfully!")
    print("Contents:", os.listdir(extract_path)[:20])


TRAIN extracted successfully!
Contents: ['train']

TEST extracted successfully!
Contents: ['test']


In [4]:
from pathlib import Path
from collections import Counter

for split in ["train", "test"]:
    root = Path(f"/content/datasets/{split}")

    images = [
        p for p in root.rglob("*")
        if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
    ]

    print(f"\n{split.upper()}")
    print("Total images:", len(images))
    print("Example paths:")
    for p in images[:5]:
        print(p.relative_to(root))

    print("Immediate folders:",
          [p.name for p in root.iterdir() if p.is_dir()][:20])


TRAIN
Total images: 2400
Example paths:
train/Office/image_0098.jpg
train/Office/image_0198.jpg
train/Office/image_0032.jpg
train/Office/image_0067.jpg
train/Office/image_0087.jpg
Immediate folders: ['train']

TEST
Total images: 400
Example paths:
test/Office/image_0089.jpg
test/Office/image_0110.jpg
test/Office/image_0112.jpg
test/Office/image_0120.jpg
test/Office/image_0130.jpg
Immediate folders: ['test']


In [5]:
import torch
import numpy as np
import random

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

train_dir = "/content/datasets/train/train"
test_dir = "/content/datasets/test/test"

train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

eval_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

full_train = datasets.ImageFolder(train_dir)
labels = np.array(full_train.targets)

train_idx, val_idx = train_test_split(
    np.arange(len(labels)),
    test_size=0.15,
    stratify=labels,
    random_state=42
)

train_dataset = Subset(
    datasets.ImageFolder(train_dir, transform=train_transform),
    train_idx
)

val_dataset = Subset(
    datasets.ImageFolder(train_dir, transform=eval_transform),
    val_idx
)

test_dataset = datasets.ImageFolder(
    test_dir, transform=eval_transform
)

assert full_train.classes == test_dataset.classes, \
    "Train and test class names do not match!"

train_loader = DataLoader(
    train_dataset, batch_size=64, shuffle=True,
    num_workers=2, pin_memory=True
)

val_loader = DataLoader(
    val_dataset, batch_size=64, shuffle=False,
    num_workers=2, pin_memory=True
)

test_loader = DataLoader(
    test_dataset, batch_size=64, shuffle=False,
    num_workers=2, pin_memory=True
)

print("Classes:", full_train.classes)
print("Number of classes:", len(full_train.classes))
print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))

Classes: ['Bedroom', 'Coast', 'Flower', 'Forest', 'Highway', 'Industrial', 'InsideCity', 'Kitchen', 'LivingRoom', 'Mountain', 'Office', 'OpenCountry', 'Store', 'Street', 'Suburb', 'TallBuilding']
Number of classes: 16
Training samples: 2040
Validation samples: 360
Test samples: 400


In [6]:
import torch.nn as nn
import torch.optim as optim
from torchvision.models import resnet18
from tqdm.auto import tqdm
import copy

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def create_model():
    model = resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, 16)
    return model.to(device)

def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    return 100 * correct / total

baseline = create_model()

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    baseline.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

best_val = -1
best_weights = None

for epoch in range(15):
    baseline.train()
    running_loss = 0

    for images, labels in tqdm(
        train_loader, desc=f"Epoch {epoch+1}/15"
    ):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = baseline(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)

    val_acc = evaluate(baseline, val_loader)

    print(
        f"Epoch {epoch+1}: "
        f"Loss={running_loss/len(train_dataset):.4f}, "
        f"Val Accuracy={val_acc:.2f}%"
    )

    if val_acc > best_val:
        best_val = val_acc
        best_weights = copy.deepcopy(baseline.state_dict())

baseline.load_state_dict(best_weights)

torch.save({
    "model_state_dict": baseline.state_dict(),
    "classes": full_train.classes,
    "validation_accuracy": best_val
}, "/content/baseline_best.pth")

print("Best validation accuracy:", best_val)
print("Baseline test accuracy:", evaluate(baseline, test_loader))

Epoch 1/15:   0%|          | 0/32 [00:00<?, ?it/s]

Epoch 1: Loss=2.2061, Val Accuracy=16.11%


Epoch 2/15:   0%|          | 0/32 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7c8b6a7360c0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
Exception ignored in:     <function _MultiProcessingDataLoaderIter.__del__ at 0x7c8b6a7360c0>self._shutdown_workers()

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
        if w.is_alive():self._shutdown_workers()

  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
        if w.is_alive():assert self._parent_pid == os.getpid(), 'can only test a child process'

AssertionError  File "/usr/lib/python3.13/multiprocessing/proce

Epoch 2: Loss=1.6793, Val Accuracy=43.61%


Epoch 3/15:   0%|          | 0/32 [00:00<?, ?it/s]

... (33 outputs left out)

Epoch 15/15:   0%|          | 0/32 [00:00<?, ?it/s]

Epoch 15: Loss=0.4061, Val Accuracy=63.61%
Best validation accuracy: 65.55555555555556


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7c8b6a7360c0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
      File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
if w.is_alive():
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process


Baseline test accuracy: 68.0


In [7]:
train_loader = DataLoader(
    train_dataset, batch_size=64, shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset, batch_size=64, shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset, batch_size=64, shuffle=False,
    num_workers=0
)

In [8]:
from torchvision import transforms, datasets
from torch.utils.data import Dataset, DataLoader
from PIL import Image

ssl_transform = transforms.Compose([
    transforms.RandomResizedCrop(128, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomApply([
        transforms.ColorJitter(0.3, 0.3, 0.3, 0.1)
    ], p=0.8),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

class SimCLRDataset(Dataset):
    def __init__(self, imagefolder, indices, transform):
        self.samples = [imagefolder.samples[i] for i in indices]
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        path, _ = self.samples[index]
        with Image.open(path) as img:
            image = img.convert("RGB")
            return self.transform(image), self.transform(image)

ssl_dataset = SimCLRDataset(full_train, train_idx, ssl_transform)

ssl_loader = DataLoader(
    ssl_dataset, batch_size=64, shuffle=True,
    drop_last=True, num_workers=0
)

print("SSL images:", len(ssl_dataset))

SSL images: 2040


In [9]:
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet18

class SimCLR(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = resnet18(weights=None)
        self.encoder.fc = nn.Identity()

        self.projector = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 128)
        )

    def forward(self, x):
        features = self.encoder(x)
        return F.normalize(self.projector(features), dim=1)

def nt_xent(z1, z2, temperature=0.5):
    batch_size = z1.size(0)
    z = torch.cat([z1, z2], dim=0)

    logits = z @ z.T / temperature
    logits.fill_diagonal_(-1e9)

    targets = (
        torch.arange(2 * batch_size, device=z.device)
        + batch_size
    ) % (2 * batch_size)

    return F.cross_entropy(logits, targets)

simclr = SimCLR().to(device)
ssl_optimizer = torch.optim.AdamW(
    simclr.parameters(), lr=0.001, weight_decay=1e-4
)

In [10]:
from tqdm.auto import tqdm

ssl_losses = []

for epoch in range(30):
    simclr.train()
    total_loss = 0

    for view1, view2 in tqdm(
        ssl_loader, desc=f"SSL Epoch {epoch+1}/30"
    ):
        view1 = view1.to(device)
        view2 = view2.to(device)

        ssl_optimizer.zero_grad()

        z1 = simclr(view1)
        z2 = simclr(view2)

        loss = nt_xent(z1, z2)
        loss.backward()
        ssl_optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(ssl_loader)
    ssl_losses.append(avg_loss)
    print(f"SSL Epoch {epoch+1}: Loss={avg_loss:.4f}")

torch.save(simclr.encoder.state_dict(), "/content/ssl_encoder.pth")
print("SSL pretraining completed!")

SSL Epoch 1/30:   0%|          | 0/31 [00:00<?, ?it/s]

SSL Epoch 1: Loss=4.1913


SSL Epoch 2/30:   0%|          | 0/31 [00:00<?, ?it/s]

SSL Epoch 2: Loss=3.8653


SSL Epoch 3/30:   0%|          | 0/31 [00:00<?, ?it/s]

SSL Epoch 3: Loss=3.6537


... (50 outputs left out)

SSL Epoch 29/30:   0%|          | 0/31 [00:00<?, ?it/s]

SSL Epoch 29: Loss=3.1491


SSL Epoch 30/30:   0%|          | 0/31 [00:00<?, ?it/s]

SSL Epoch 30: Loss=3.1517
SSL pretraining completed!


In [11]:
import copy

ssl_classifier = create_model()

encoder_weights = torch.load(
    "/content/ssl_encoder.pth",
    map_location=device,
    weights_only=True
)

ssl_classifier.load_state_dict(encoder_weights, strict=False)

optimizer = torch.optim.AdamW(
    ssl_classifier.parameters(),
    lr=0.0003,
    weight_decay=1e-4
)

criterion = nn.CrossEntropyLoss()
best_ssl_val = -1
best_ssl_weights = None

for epoch in range(15):
    ssl_classifier.train()

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = ssl_classifier(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    val_acc = evaluate(ssl_classifier, val_loader)
    print(f"Fine-tune Epoch {epoch+1}: Val={val_acc:.2f}%")

    if val_acc > best_ssl_val:
        best_ssl_val = val_acc
        best_ssl_weights = copy.deepcopy(
            ssl_classifier.state_dict()
        )

ssl_classifier.load_state_dict(best_ssl_weights)

torch.save({
    "model_state_dict": ssl_classifier.state_dict(),
    "classes": full_train.classes,
    "validation_accuracy": best_ssl_val
}, "/content/ssl_best.pth")

print("SSL best validation accuracy:", best_ssl_val)
print("SSL test accuracy:", evaluate(ssl_classifier, test_loader))

Fine-tune Epoch 1: Val=48.06%
Fine-tune Epoch 2: Val=56.94%
Fine-tune Epoch 3: Val=58.89%
Fine-tune Epoch 4: Val=66.11%
Fine-tune Epoch 5: Val=63.33%
Fine-tune Epoch 6: Val=65.56%
Fine-tune Epoch 7: Val=63.06%
Fine-tune Epoch 8: Val=63.89%
Fine-tune Epoch 9: Val=64.44%
Fine-tune Epoch 10: Val=67.78%
Fine-tune Epoch 11: Val=64.44%
Fine-tune Epoch 12: Val=70.00%
Fine-tune Epoch 13: Val=66.39%
Fine-tune Epoch 14: Val=66.39%
Fine-tune Epoch 15: Val=68.06%
SSL best validation accuracy: 70.0
SSL test accuracy: 70.5


In [12]:
import torch
import torch.nn as nn
import copy

improved_model = create_model()

checkpoint = torch.load(
    "/content/ssl_best.pth",
    map_location=device,
    weights_only=True
)

improved_model.load_state_dict(checkpoint["model_state_dict"])

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.AdamW(
    improved_model.parameters(),
    lr=0.00005,
    weight_decay=0.001
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=20
)

best_improved_val = evaluate(improved_model, val_loader)
best_improved_weights = copy.deepcopy(improved_model.state_dict())

print("Starting validation accuracy:", best_improved_val)

for epoch in range(20):
    improved_model.train()

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = improved_model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

    scheduler.step()

    val_acc = evaluate(improved_model, val_loader)

    print(f"Improved Epoch {epoch+1}/20: Val={val_acc:.2f}%")

    if val_acc > best_improved_val:
        best_improved_val = val_acc
        best_improved_weights = copy.deepcopy(
            improved_model.state_dict()
        )

improved_model.load_state_dict(best_improved_weights)

torch.save({
    "model_state_dict": improved_model.state_dict(),
    "classes": full_train.classes,
    "validation_accuracy": best_improved_val
}, "/content/improved_best.pth")

print("Improved best validation accuracy:", best_improved_val)
print("Improved test accuracy:", evaluate(improved_model, test_loader))

Starting validation accuracy: 70.0
Improved Epoch 1/20: Val=70.83%
Improved Epoch 2/20: Val=68.89%
Improved Epoch 3/20: Val=70.28%
Improved Epoch 4/20: Val=69.17%
Improved Epoch 5/20: Val=68.61%
Improved Epoch 6/20: Val=69.44%
Improved Epoch 7/20: Val=68.89%
Improved Epoch 8/20: Val=70.56%
Improved Epoch 9/20: Val=71.67%
Improved Epoch 10/20: Val=71.11%
Improved Epoch 11/20: Val=71.94%
Improved Epoch 12/20: Val=70.56%
Improved Epoch 13/20: Val=70.56%
Improved Epoch 14/20: Val=70.56%
Improved Epoch 15/20: Val=71.11%
Improved Epoch 16/20: Val=70.83%
Improved Epoch 17/20: Val=70.28%
Improved Epoch 18/20: Val=70.83%
Improved Epoch 19/20: Val=70.00%
Improved Epoch 20/20: Val=70.56%
Improved best validation accuracy: 71.94444444444444
Improved test accuracy: 75.0


In [13]:
from google.colab import drive
import os, shutil

drive.mount("/content/drive")

save_dir = "/content/drive/MyDrive/CV_Assignment2"
os.makedirs(save_dir, exist_ok=True)

for filename in [
    "baseline_best.pth",
    "ssl_encoder.pth",
    "ssl_best.pth",
    "improved_best.pth"
]:
    shutil.copy(
        f"/content/{filename}",
        os.path.join(save_dir, filename)
    )

print("All checkpoints saved to Google Drive!")

Mounted at /content/drive
All checkpoints saved to Google Drive!


In [14]:
import pandas as pd

results = pd.DataFrame({
    "Method": [
        "Supervised ResNet18",
        "SimCLR + Fine-tuning",
        "Improved SSL + Regularization"
    ],
    "Validation Accuracy": [65.56, 70.00, 71.94],
    "Test Accuracy": [68.00, 70.50, 75.00]
})

results.to_csv(
    "/content/drive/MyDrive/CV_Assignment2/results.csv",
    index=False
)

print(results)

                          Method  Validation Accuracy  Test Accuracy
0            Supervised ResNet18                65.56           68.0
1           SimCLR + Fine-tuning                70.00           70.5
2  Improved SSL + Regularization                71.94           75.0


In [15]:
import torch
import torch.nn.functional as F

@torch.no_grad()
def evaluate_tta(model, loader):
    model.eval()
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        original_logits = model(images)
        flipped_logits = model(torch.flip(images, dims=[3]))

        original_probs = F.softmax(original_logits, dim=1)
        flipped_probs = F.softmax(flipped_logits, dim=1)

        avg_probs = (original_probs + flipped_probs) / 2
        predictions = avg_probs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return 100 * correct / total

print("Original validation accuracy:",
      evaluate(improved_model, val_loader))

print("TTA validation accuracy:",
      evaluate_tta(improved_model, val_loader))

Original validation accuracy: 71.94444444444444
TTA validation accuracy: 72.22222222222223


In [16]:
original_val = evaluate(improved_model, val_loader)
tta_val = evaluate_tta(improved_model, val_loader)

if tta_val > original_val:
    selected = "TTA"
    final_test = evaluate_tta(improved_model, test_loader)
else:
    selected = "Original"
    final_test = evaluate(improved_model, test_loader)

print("Selected inference:", selected)
print("Selected test accuracy:", final_test)

Selected inference: TTA
Selected test accuracy: 75.25


In [17]:
import numpy as np
import torch
import torch.nn as nn
import copy

mixup_model = create_model()

checkpoint = torch.load(
    "/content/improved_best.pth",
    map_location=device,
    weights_only=True
)

mixup_model.load_state_dict(checkpoint["model_state_dict"])

optimizer = torch.optim.AdamW(
    mixup_model.parameters(),
    lr=0.00003,
    weight_decay=0.001
)

criterion = nn.CrossEntropyLoss()

def mixup_data(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha)
    index = torch.randperm(x.size(0), device=x.device)

    mixed_x = lam * x + (1 - lam) * x[index]

    return mixed_x, y, y[index], lam

best_mixup_val = evaluate(mixup_model, val_loader)
best_mixup_weights = copy.deepcopy(mixup_model.state_dict())

print("Starting validation:", best_mixup_val)

for epoch in range(12):
    mixup_model.train()

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        mixed, labels_a, labels_b, lam = mixup_data(
            images, labels, alpha=0.2
        )

        optimizer.zero_grad()

        outputs = mixup_model(mixed)

        loss = (
            lam * criterion(outputs, labels_a)
            + (1 - lam) * criterion(outputs, labels_b)
        )

        loss.backward()
        optimizer.step()

    val_acc = evaluate(mixup_model, val_loader)
    print(f"Mixup Epoch {epoch+1}/12: Val={val_acc:.2f}%")

    if val_acc > best_mixup_val:
        best_mixup_val = val_acc
        best_mixup_weights = copy.deepcopy(
            mixup_model.state_dict()
        )

mixup_model.load_state_dict(best_mixup_weights)

torch.save({
    "model_state_dict": mixup_model.state_dict(),
    "classes": full_train.classes,
    "validation_accuracy": best_mixup_val
}, "/content/mixup_best.pth")

print("Best Mixup validation accuracy:", best_mixup_val)

Starting validation: 71.94444444444444
Mixup Epoch 1/12: Val=70.56%
Mixup Epoch 2/12: Val=70.00%
Mixup Epoch 3/12: Val=70.28%
Mixup Epoch 4/12: Val=71.94%
Mixup Epoch 5/12: Val=70.56%
Mixup Epoch 6/12: Val=71.11%
Mixup Epoch 7/12: Val=71.67%
Mixup Epoch 8/12: Val=71.94%
Mixup Epoch 9/12: Val=71.94%
Mixup Epoch 10/12: Val=73.06%
Mixup Epoch 11/12: Val=70.56%
Mixup Epoch 12/12: Val=71.67%
Best Mixup validation accuracy: 73.05555555555556


In [18]:
import numpy as np
import torch
import torch.nn as nn
import copy

mixup_model = create_model()

checkpoint = torch.load(
    "/content/improved_best.pth",
    map_location=device,
    weights_only=True
)

mixup_model.load_state_dict(checkpoint["model_state_dict"])

optimizer = torch.optim.AdamW(
    mixup_model.parameters(),
    lr=0.00003,
    weight_decay=0.001
)

criterion = nn.CrossEntropyLoss()

def mixup_data(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha)
    index = torch.randperm(x.size(0), device=x.device)

    mixed_x = lam * x + (1 - lam) * x[index]

    return mixed_x, y, y[index], lam

best_mixup_val = evaluate(mixup_model, val_loader)
best_mixup_weights = copy.deepcopy(mixup_model.state_dict())

print("Starting validation:", best_mixup_val)

for epoch in range(12):
    mixup_model.train()

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        mixed, labels_a, labels_b, lam = mixup_data(
            images, labels, alpha=0.2
        )

        optimizer.zero_grad()

        outputs = mixup_model(mixed)

        loss = (
            lam * criterion(outputs, labels_a)
            + (1 - lam) * criterion(outputs, labels_b)
        )

        loss.backward()
        optimizer.step()

    val_acc = evaluate(mixup_model, val_loader)
    print(f"Mixup Epoch {epoch+1}/12: Val={val_acc:.2f}%")

    if val_acc > best_mixup_val:
        best_mixup_val = val_acc
        best_mixup_weights = copy.deepcopy(
            mixup_model.state_dict()
        )

mixup_model.load_state_dict(best_mixup_weights)

torch.save({
    "model_state_dict": mixup_model.state_dict(),
    "classes": full_train.classes,
    "validation_accuracy": best_mixup_val
}, "/content/mixup_best.pth")

print("Best Mixup validation accuracy:", best_mixup_val)

Starting validation: 71.94444444444444
Mixup Epoch 1/12: Val=70.28%
Mixup Epoch 2/12: Val=71.39%
Mixup Epoch 3/12: Val=70.00%
Mixup Epoch 4/12: Val=71.94%
Mixup Epoch 5/12: Val=71.11%
Mixup Epoch 6/12: Val=71.39%
Mixup Epoch 7/12: Val=70.28%
Mixup Epoch 8/12: Val=73.06%
Mixup Epoch 9/12: Val=71.67%
Mixup Epoch 10/12: Val=73.33%
Mixup Epoch 11/12: Val=71.67%
Mixup Epoch 12/12: Val=70.56%
Best Mixup validation accuracy: 73.33333333333333


In [19]:
original_val = evaluate(mixup_model, val_loader)
tta_val = evaluate_tta(mixup_model, val_loader)

print("Mixup original validation:", original_val)
print("Mixup TTA validation:", tta_val)

Mixup original validation: 73.33333333333333
Mixup TTA validation: 71.11111111111111


In [20]:
mixup_test_accuracy = evaluate(mixup_model, test_loader)

print(f"Mixup validation accuracy: {best_mixup_val:.2f}%")
print(f"Mixup test accuracy: {mixup_test_accuracy:.2f}%")

Mixup validation accuracy: 73.33%
Mixup test accuracy: 73.75%


In [21]:
import os
import shutil
from google.colab import drive

drive.mount("/content/drive")

save_dir = "/content/drive/MyDrive/CV_Assignment2"
os.makedirs(save_dir, exist_ok=True)

for filename in [
    "baseline_best.pth",
    "ssl_encoder.pth",
    "ssl_best.pth",
    "improved_best.pth",
    "mixup_best.pth"
]:
    source = f"/content/{filename}"
    if os.path.exists(source):
        shutil.copy2(source, os.path.join(save_dir, filename))
        print("Saved:", filename)
    else:
        print("Missing:", filename)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved: baseline_best.pth
Saved: ssl_encoder.pth
Saved: ssl_best.pth
Saved: improved_best.pth
Saved: mixup_best.pth


In [22]:
mixup_test_accuracy = evaluate(mixup_model, test_loader)

print(f"Mixup validation: {best_mixup_val:.2f}%")
print(f"Mixup test: {mixup_test_accuracy:.2f}%")

Mixup validation: 73.33%
Mixup test: 73.75%


In [23]:
from google.colab import drive
import os
import shutil

drive.mount("/content/drive")

save_dir = "/content/drive/MyDrive/CV_Assignment2"
os.makedirs(save_dir, exist_ok=True)

files = [
    "baseline_best.pth",
    "ssl_encoder.pth",
    "ssl_best.pth",
    "improved_best.pth",
    "mixup_best.pth"
]

for filename in files:
    source = f"/content/{filename}"
    destination = os.path.join(save_dir, filename)

    if os.path.exists(source):
        shutil.copy2(source, destination)
        print("Saved:", filename)
    else:
        print("MISSING:", filename)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved: baseline_best.pth
Saved: ssl_encoder.pth
Saved: ssl_best.pth
Saved: improved_best.pth
Saved: mixup_best.pth
